# Pydantic — Nível Avançado

**Objetivo:** dominar os recursos que aparecem em sistemas grandes: payloads polimórficos, contratos genéricos,
validação sem classe, tipos de domínio próprios, validação dependente de contexto, serialização sob medida,
JSON Schema como contrato, performance e evolução de versões.

Roteiro:

1. Uniões discriminadas (webhooks / eventos)
2. Modelos genéricos (`Page[T]`, `ApiResponse[T]`)
3. `TypeAdapter` e `RootModel`
4. Tipos de domínio customizados (`Annotated` e `__get_pydantic_core_schema__`)
5. Validators `wrap` e **contexto** de validação
6. Serialização sob medida (`field_serializer`, `model_serializer`, contexto de serialização)
7. Erros customizados (`PydanticCustomError`)
8. JSON Schema como contrato
9. `@validate_call`: fronteira em funções
10. JSON parcial (streaming)
11. Performance: medindo os caminhos
12. Evolução de contrato (v1 → v2)
13. Checklist de produção

In [1]:
import json, timeit
from datetime import datetime, timezone
from decimal import Decimal
from typing import Annotated, Any, Generic, Literal, TypeVar, Union
from pydantic import (BaseModel, ConfigDict, Field, TypeAdapter, ValidationError,
                      field_validator, model_validator, field_serializer, model_serializer)

def show_errors(e: ValidationError) -> None:
    for err in e.errors(include_url=False):
        print(f"  {'.'.join(map(str, err['loc'])) or '<model>':<34} {err['type']:<24} {err['msg']}")

## 1. Uniões discriminadas

Webhooks, eventos de fila e mensagens de chat costumam ser **polimórficos**: o mesmo endpoint recebe formas diferentes,
identificadas por um campo (`type`, `kind`, `event`).

Com `A | B | C` simples, o Pydantic tenta encaixar em cada um (modo *smart*): é mais lento e o erro lista falhas de **todos** os tipos.
Com `discriminator`, ele lê o campo, vai **direto** no modelo certo e o erro é preciso.

In [2]:
class PaymentApproved(BaseModel):
    type: Literal["payment.approved"]
    payment_id: str
    amount: Decimal

class PaymentRefunded(BaseModel):
    type: Literal["payment.refunded"]
    payment_id: str
    refund_amount: Decimal
    reason: str

class CustomerCreated(BaseModel):
    type: Literal["customer.created"]
    customer_id: str
    email: str

WebhookEvent = Annotated[
    Union[PaymentApproved, PaymentRefunded, CustomerCreated],
    Field(discriminator="type"),
]

class WebhookEnvelope(BaseModel):
    id: str
    created_at: datetime
    data: WebhookEvent

raw_events = [
    '{"id": "evt_1", "created_at": "2026-11-10T10:00:00Z", "data": {"type": "payment.approved", "payment_id": "p1", "amount": "99.90"}}',
    '{"id": "evt_2", "created_at": "2026-11-10T10:05:00Z", "data": {"type": "customer.created", "customer_id": "c9", "email": "a@b.c"}}',
]
for raw in raw_events:
    ev = WebhookEnvelope.model_validate_json(raw)
    print(f"{type(ev.data).__name__} -> {ev.data}")

PaymentApproved -> type='payment.approved' payment_id='p1' amount=Decimal('99.90')
CustomerCreated -> type='customer.created' customer_id='c9' email='a@b.c'


In [3]:
# Despacho por tipo com pattern matching: o type checker sabe qual classe está em cada case
def handle(ev: WebhookEnvelope) -> str:
    match ev.data:
        case PaymentApproved(amount=amount):
            return f"liberar pedido, valor {amount}"
        case PaymentRefunded(reason=reason):
            return f"estornar: {reason}"
        case CustomerCreated(email=email):
            return f"enviar boas-vindas para {email}"

print(f"[handle(WebhookEnvelope.model_validate_json(r)) for r in raw_events]: {[handle(WebhookEnvelope.model_validate_json(r)) for r in raw_events]}")

# erro preciso: só reclama do modelo escolhido pelo discriminador
try:
    WebhookEnvelope.model_validate_json('{"id": "e", "created_at": "2026-11-10T10:00:00Z", "data": {"type": "payment.refunded", "payment_id": "p1"}}')
except ValidationError as e:
    show_errors(e)
try:
    WebhookEnvelope.model_validate_json('{"id": "e", "created_at": "2026-11-10T10:00:00Z", "data": {"type": "invoice.paid"}}')
except ValidationError as e:
    show_errors(e)

[handle(WebhookEnvelope.model_validate_json(r)) for r in raw_events]: ['liberar pedido, valor 99.90', 'enviar boas-vindas para a@b.c']
  data.payment.refunded.refund_amount missing                  Field required
  data.payment.refunded.reason       missing                  Field required
  data                               union_tag_invalid        Input tag 'invoice.paid' found using 'type' does not match any of the expected tags: 'payment.approved', 'payment.refunded', 'customer.created'


### Discriminador calculado
Quando o payload não tem um campo de tipo explícito (ou ele está em lugares diferentes), use um `Discriminator` com função + `Tag`:

In [4]:
from pydantic import Discriminator, Tag

class CardPayment(BaseModel):
    card_last4: str
    brand: str

class PixPayment(BaseModel):
    pix_key: str

def payment_kind(v: Any) -> str:
    data = v if isinstance(v, dict) else v.__dict__
    return "card" if "card_last4" in data else "pix"

PaymentMethod = Annotated[
    Union[Annotated[CardPayment, Tag("card")], Annotated[PixPayment, Tag("pix")]],
    Discriminator(payment_kind),
]
ta = TypeAdapter(list[PaymentMethod])
methods = [{"card_last4": "4242", "brand": "visa"}, {"pix_key": "dante@pix"}]
print(f"ta.validate_python(methods): {ta.validate_python(methods)}")

ta.validate_python(methods): [CardPayment(card_last4='4242', brand='visa'), PixPayment(pix_key='dante@pix')]


## 2. Modelos genéricos

Todo endpoint de listagem devolve "página de alguma coisa"; toda resposta tem o mesmo envelope. Com genéricos,
você escreve o envelope **uma vez** e o JSON Schema/OpenAPI de cada especialização sai correto (`Page[User]`, `Page[Order]`).

In [5]:
T = TypeVar("T")

class Page(BaseModel, Generic[T]):
    items: list[T]
    total: int = Field(ge=0)
    page: int = Field(ge=1)
    size: int = Field(ge=1, le=100)

    @property
    def pages(self) -> int:
        return -(-self.total // self.size)

class ApiError(BaseModel):
    code: str
    message: str

class ApiResponse(BaseModel, Generic[T]):
    ok: bool
    data: T | None = None
    error: ApiError | None = None

    @model_validator(mode="after")
    def data_xor_error(self):
        if self.ok == (self.data is None) or self.ok == (self.error is not None):
            raise ValueError("ok=True exige data e proíbe error; ok=False exige error")
        return self

class User(BaseModel):
    id: int
    name: str

resp = ApiResponse[Page[User]].model_validate({
    "ok": True,
    "data": {"items": [{"id": 1, "name": "Ana"}, {"id": "2", "name": "Bia"}], "total": 42, "page": 1, "size": 2},
})
print(f"resp.data.items: {resp.data.items}")
print(f"páginas: {resp.data.pages}")
print(f"ApiResponse[Page[User]].__name__: {ApiResponse[Page[User]].__name__}")

try:
    ApiResponse[Page[User]].model_validate({"ok": True, "data": {"items": [{"id": "x"}], "total": 1, "page": 1, "size": 10}})
except ValidationError as e:
    show_errors(e)

resp.data.items: [User(id=1, name='Ana'), User(id=2, name='Bia')]
páginas: 21
ApiResponse[Page[User]].__name__: ApiResponse[Page[User]]
  data.items.0.id                    int_parsing              Input should be a valid integer, unable to parse string as an integer
  data.items.0.name                  missing                  Field required


## 3. `TypeAdapter` e `RootModel`

Nem tudo precisa de classe. `TypeAdapter` valida/serializa/gera schema para **qualquer tipo**: `list[User]`,
`dict[str, int]`, `TypedDict`, `dataclass`, uniões...

**Regra de performance**: criar um `TypeAdapter` monta o schema (caro). Crie **uma vez** no módulo e reutilize.

In [6]:
from typing import TypedDict

class Metric(TypedDict):
    name: str
    value: float

UsersAdapter = TypeAdapter(list[User])                  # nível de módulo
MetricsAdapter = TypeAdapter(dict[str, list[Metric]])

users = UsersAdapter.validate_json('[{"id": 1, "name": "Ana"}, {"id": 2, "name": "Bia"}]')
print(f"users: {users}")
print(f"UsersAdapter.dump_json(users): {UsersAdapter.dump_json(users)}")

metrics = MetricsAdapter.validate_python({"cpu": [{"name": "p95", "value": "0.83"}]})
print(f"metrics: {metrics}")
print(f'type(metrics["cpu"][0]): {type(metrics["cpu"][0])}')                  # TypedDict continua sendo dict
print(f"json.dumps(MetricsAdapter.json_schema())[:160]: {json.dumps(MetricsAdapter.json_schema())[:160]} ...")

users: [User(id=1, name='Ana'), User(id=2, name='Bia')]
UsersAdapter.dump_json(users): b'[{"id":1,"name":"Ana"},{"id":2,"name":"Bia"}]'
metrics: {'cpu': [{'name': 'p95', 'value': 0.83}]}
type(metrics["cpu"][0]): <class 'dict'>
json.dumps(MetricsAdapter.json_schema())[:160]: {"$defs": {"Metric": {"properties": {"name": {"title": "Name", "type": "string"}, "value": {"title": "Value", "type": "number"}}, "required": ["name", "value"], ...


`RootModel` é para quando o **JSON inteiro** é uma lista/valor e você quer métodos/serialização de modelo:

In [7]:
from pydantic import RootModel

class Tags(RootModel[list[str]]):
    @field_validator("root")
    @classmethod
    def normalize(cls, v: list[str]) -> list[str]:
        return sorted({t.strip().lower() for t in v if t.strip()})

    def __iter__(self):
        return iter(self.root)

tags = Tags.model_validate_json('["Python", " pydantic ", "python", ""]')
print(f"tags.root: {tags.root}")
print(f"list(tags): {list(tags)}")
print(f"tags.model_dump_json(): {tags.model_dump_json()}")

tags.root: ['pydantic', 'python']
list(tags): ['pydantic', 'python']
tags.model_dump_json(): ["pydantic","python"]


## 4. Tipos de domínio customizados

### 4.1 Via `Annotated` (resolve 90% dos casos)
Entrada flexível + armazenamento canônico + saída controlada:

In [8]:
from pydantic import BeforeValidator, PlainSerializer, WithJsonSchema

def _parse_money(v: Any) -> Decimal:
    if isinstance(v, str):
        v = v.replace("R$", "").strip()
        if "," in v:
            v = v.replace(".", "").replace(",", ".")
    d = Decimal(str(v))
    return d.quantize(Decimal("0.01"))

BRL = Annotated[
    Decimal,
    BeforeValidator(_parse_money),
    Field(ge=0),
    PlainSerializer(lambda d: f"R$ {d:,.2f}".replace(",", "X").replace(".", ",").replace("X", "."),
                    return_type=str, when_used="json"),
    WithJsonSchema({"type": "string", "examples": ["R$ 1.234,56"]}, mode="serialization"),
]

class Quote(BaseModel):
    product: str
    price: BRL

q = Quote(product="Notebook", price="R$ 3.499,90")
print(f"repr(q.price): {repr(q.price)}")                 # Python: Decimal canônico
print(f"q.model_dump_json(): {q.model_dump_json()}")           # JSON: formato humano
print(f'Quote(product="x", price=10).price: {Quote(product="x", price=10).price}')

repr(q.price): Decimal('3499.90')
q.model_dump_json(): {"product":"Notebook","price":"R$ 3.499,90"}
Quote(product="x", price=10).price: 10.00


### 4.2 Via `__get_pydantic_core_schema__` (classe própria)
Quando você quer um **tipo de verdade** (com métodos), que valida a si mesmo onde quer que seja usado:

In [9]:
from pydantic import GetCoreSchemaHandler
from pydantic_core import core_schema

class GeoPoint:
    """Aceita 'lat,lon', [lat, lon] ou {'lat':..,'lon':..}; serializa como 'lat,lon'."""
    __slots__ = ("lat", "lon")

    def __init__(self, lat: float, lon: float):
        if not (-90 <= lat <= 90 and -180 <= lon <= 180):
            raise ValueError(f"coordenada fora do intervalo: {lat},{lon}")
        self.lat, self.lon = lat, lon

    def __repr__(self):
        return f"GeoPoint({self.lat}, {self.lon})"

    @classmethod
    def _parse(cls, v: Any) -> "GeoPoint":
        if isinstance(v, GeoPoint):
            return v
        if isinstance(v, str):
            lat, lon = (float(x) for x in v.split(","))
        elif isinstance(v, (list, tuple)) and len(v) == 2:
            lat, lon = map(float, v)
        elif isinstance(v, dict):
            lat, lon = float(v["lat"]), float(v["lon"])
        else:
            raise ValueError("formato de coordenada não suportado")
        return cls(lat, lon)

    @classmethod
    def __get_pydantic_core_schema__(cls, source: Any, handler: GetCoreSchemaHandler):
        return core_schema.no_info_plain_validator_function(
            cls._parse,
            serialization=core_schema.plain_serializer_function_ser_schema(lambda p: f"{p.lat},{p.lon}"),
        )

class Store(BaseModel):
    name: str
    location: GeoPoint

for loc in ["-8.05,-34.88", [-23.55, -46.63], {"lat": -15.79, "lon": -47.88}]:
    s = Store(name="loja", location=loc)
    print(f"{s.location} -> {s.model_dump_json()}")
try:
    Store(name="x", location="200,10")
except ValidationError as e:
    show_errors(e)

GeoPoint(-8.05, -34.88) -> {"name":"loja","location":"-8.05,-34.88"}
GeoPoint(-23.55, -46.63) -> {"name":"loja","location":"-23.55,-46.63"}
GeoPoint(-15.79, -47.88) -> {"name":"loja","location":"-15.79,-47.88"}
  location                           value_error              Value error, coordenada fora do intervalo: 200.0,10.0


## 5. Validators `wrap` e contexto de validação

### 5.1 `wrap`: você controla a validação padrão
Útil para **fallback**: tenta validar; se falhar, aplica um plano B em vez de rejeitar.

In [10]:
from pydantic import WrapValidator, ValidatorFunctionWrapHandler

def lenient_int(v: Any, handler: ValidatorFunctionWrapHandler) -> int:
    try:
        return handler(v)
    except ValidationError:
        return 0              # dado de telemetria ruim não deve derrubar o lote

TelemetryInt = Annotated[int, WrapValidator(lenient_int)]

class Sample(BaseModel):
    sensor: str
    reading: TelemetryInt

print(f'[Sample(sensor="s", reading=r).reading for r in ["10", "NaN", None, 7.0]]: {[Sample(sensor="s", reading=r).reading for r in ["10", "NaN", None, 7.0]]}')

[Sample(sensor="s", reading=r).reading for r in ["10", "NaN", None, 7.0]]: [10, 0, 0, 7]


### 5.2 Contexto: validação que depende de quem está chamando
`model_validate(data, context={...})` injeta informação externa (usuário, tenant, feature flag) que os validators leem em `info.context`.
O modelo continua puro e testável — sem variável global.

In [11]:
from pydantic import ValidationInfo

class DiscountRequest(BaseModel):
    order_id: str
    discount_pct: float = Field(ge=0, le=100)

    @field_validator("discount_pct")
    @classmethod
    def respect_role_limit(cls, v: float, info: ValidationInfo) -> float:
        ctx = info.context or {}
        limit = {"seller": 10, "manager": 30, "director": 100}.get(ctx.get("role"), 0)
        if v > limit:
            raise ValueError(f"perfil {ctx.get('role')!r} pode dar no máximo {limit}%")
        return v

data = {"order_id": "o-1", "discount_pct": 25}
print(f'DiscountRequest.model_validate(data, context={{"role": "manager"}}): {DiscountRequest.model_validate(data, context={"role": "manager"})}')
try:
    DiscountRequest.model_validate(data, context={"role": "seller"})
except ValidationError as e:
    show_errors(e)

DiscountRequest.model_validate(data, context={"role": "manager"}): order_id='o-1' discount_pct=25.0
  discount_pct                       value_error              Value error, perfil 'seller' pode dar no máximo 10%


## 6. Serialização sob medida

* `@field_serializer("campo")` → controla a saída de um campo.
* `@model_serializer(mode="wrap")` → pós-processa o dict inteiro (envelopes, campos condicionais).
* `model_dump(context=...)` → a serialização também recebe contexto (ex.: mascarar dados por perfil — LGPD).

In [12]:
from pydantic import SerializationInfo, SerializerFunctionWrapHandler

class Patient(BaseModel):
    name: str
    cpf: str
    birth: datetime
    notes: str | None = None

    @field_serializer("birth")
    def iso_date_only(self, v: datetime) -> str:
        return v.date().isoformat()

    @field_serializer("cpf")
    def mask_cpf(self, v: str, info: SerializationInfo) -> str:
        if (info.context or {}).get("can_see_pii"):
            return v
        return f"***.{v[3:6]}.***-**"

    @model_serializer(mode="wrap")
    def envelope(self, handler: SerializerFunctionWrapHandler, info: SerializationInfo) -> dict:
        data = handler(self)
        if not (info.context or {}).get("can_see_pii"):
            data.pop("notes", None)
        return {"kind": "patient", "data": data}

p = Patient(name="Maria", cpf="12345678909", birth=datetime(1990, 3, 4, 8, 30), notes="alergia a dipirona")
print(f"p.model_dump_json(): {p.model_dump_json()}")
print(f'p.model_dump_json(context={{"can_see_pii": True}}): {p.model_dump_json(context={"can_see_pii": True})}')

p.model_dump_json(): {"kind":"patient","data":{"name":"Maria","cpf":"***.456.***-**","birth":"1990-03-04"}}
p.model_dump_json(context={"can_see_pii": True}): {"kind":"patient","data":{"name":"Maria","cpf":"12345678909","birth":"1990-03-04","notes":"alergia a dipirona"}}


## 7. Erros customizados com `PydanticCustomError`

`ValueError` gera sempre `type="value_error"`. Para a API (ou o front) reagir a erros específicos, emita um **código próprio** com contexto:

In [13]:
from pydantic_core import PydanticCustomError

class Booking(BaseModel):
    guests: int
    room_capacity: int

    @model_validator(mode="after")
    def capacity(self):
        if self.guests > self.room_capacity:
            raise PydanticCustomError(
                "room_over_capacity",
                "Quarto comporta {capacity} pessoas, recebeu {guests}",
                {"capacity": self.room_capacity, "guests": self.guests},
            )
        return self

try:
    Booking(guests=5, room_capacity=3)
except ValidationError as e:
    err = e.errors(include_url=False)[0]
    print(f'err["type"]: {err["type"]}')
    print(f'err["msg"]: {err["msg"]}')
    print(f'err["ctx"]: {err["ctx"]}')

err["type"]: room_over_capacity
err["msg"]: Quarto comporta 3 pessoas, recebeu 5
err["ctx"]: {'capacity': 3, 'guests': 5}


## 8. JSON Schema como contrato

`model_json_schema()` gera JSON Schema (Draft 2020-12). Ele é a ponte com o resto do mundo:
OpenAPI/Swagger, geração de clientes TypeScript, validação no front, contratos entre microsserviços e
**structured output / tool calling de LLMs** (veja `AI_use_cases`).

* `mode="validation"` (padrão) descreve o que o modelo **aceita**; `mode="serialization"` descreve o que ele **emite** (inclui `computed_field`).
* `description`, `examples`, `title` e docstrings viram documentação.
* `json_schema_extra` adiciona chaves livres.

In [14]:
class CreateTicket(BaseModel):
    """Abre um chamado de suporte."""
    model_config = ConfigDict(json_schema_extra={"x-owner": "support-team"})

    title: str = Field(min_length=5, max_length=120, description="Resumo do problema")
    priority: Literal["low", "medium", "high"] = Field("medium", description="Urgência percebida pelo cliente")
    tags: list[str] = Field(default_factory=list, max_length=5, examples=[["login", "mobile"]])

print(f"json.dumps(CreateTicket.model_json_schema(), indent=2, ensure_ascii=False):\n{json.dumps(CreateTicket.model_json_schema(), indent=2, ensure_ascii=False)}")

json.dumps(CreateTicket.model_json_schema(), indent=2, ensure_ascii=False):
{
  "description": "Abre um chamado de suporte.",
  "properties": {
    "title": {
      "description": "Resumo do problema",
      "maxLength": 120,
      "minLength": 5,
      "title": "Title",
      "type": "string"
    },
    "priority": {
      "default": "medium",
      "description": "Urgência percebida pelo cliente",
      "enum": [
        "low",
        "medium",
        "high"
      ],
      "title": "Priority",
      "type": "string"
    },
    "tags": {
      "examples": [
        [
          "login",
          "mobile"
        ]
      ],
      "items": {
        "type": "string"
      },
      "maxItems": 5,
      "title": "Tags",
      "type": "array"
    }
  },
  "required": [
    "title"
  ],
  "title": "CreateTicket",
  "type": "object",
  "x-owner": "support-team"
}


## 9. `@validate_call`: fronteira em funções

Valida (e converte) os **argumentos** de uma função — e opcionalmente o retorno — com as mesmas regras dos modelos.
Útil em SDKs internos, funções expostas como *tools* para agentes e scripts que recebem dados de CLI.

In [15]:
from pydantic import validate_call

@validate_call(validate_return=True)
def schedule_report(email: Annotated[str, Field(pattern=r".+@.+")],
                    hour: Annotated[int, Field(ge=0, le=23)],
                    formats: list[Literal["pdf", "csv"]] = ["pdf"]) -> dict:
    return {"to": email, "at": f"{hour:02d}:00", "formats": formats}

print(f'schedule_report("time@empresa.com", "7", ["csv", "pdf"]): {schedule_report("time@empresa.com", "7", ["csv", "pdf"])}')
try:
    schedule_report("sem-arroba", 25, ["xlsx"])
except ValidationError as e:
    show_errors(e)

schedule_report("time@empresa.com", "7", ["csv", "pdf"]): {'to': 'time@empresa.com', 'at': '07:00', 'formats': ['csv', 'pdf']}
  0                                  string_pattern_mismatch  String should match pattern '.+@.+'
  1                                  less_than_equal          Input should be less than or equal to 23
  2.0                                literal_error            Input should be 'pdf' or 'csv'


## 10. JSON parcial (streaming)

APIs de streaming (incluindo LLMs) entregam JSON **aos pedaços**. `experimental_allow_partial` valida o que já chegou,
descartando o fragmento incompleto do final — dá para mostrar resultados progressivos na UI.

> Limitação (medida no 2.13): o modo parcial vale para `list`/`dict`/`TypedDict` no **topo**; ele não se propaga para dentro de
> campos de um `BaseModel`. Para objetos, use um espelho `TypedDict` durante o stream e o `BaseModel` na validação final
> (exemplo completo em `AI_use_cases/01`, seção 7).

In [16]:
class Item(BaseModel):
    name: str
    qty: int

ItemsAdapter = TypeAdapter(list[Item])
full = '[{"name": "café", "qty": 2}, {"name": "pão", "qty": 6}, {"name": "manteiga", "qty": 1}]'

for cut in (20, 45, 70, len(full)):
    chunk = full[:cut]
    parsed = ItemsAdapter.validate_json(chunk, experimental_allow_partial=True)
    print(f"{cut:>3} chars -> {[i.name for i in parsed]}")

 20 chars -> []
 45 chars -> ['café']
 70 chars -> ['café', 'pão']
 87 chars -> ['café', 'pão', 'manteiga']


Correção para quando o stream é um **objeto** (`BaseModel` com uma lista dentro): espelho `TypedDict` durante o stream,
`BaseModel` completo no final.

In [17]:
from typing import NotRequired

class Basket(BaseModel):               # contrato final
    owner: str
    items: list[Item]

class BasketStream(TypedDict):         # visão parcial
    owner: NotRequired[str]
    items: NotRequired[list[Item]]

full_obj = '{"owner": "ana", "items": [{"name": "café", "qty": 2}, {"name": "pão", "qty": 6}]}'
cut = full_obj[:60]
try:
    TypeAdapter(Basket).validate_json(cut, experimental_allow_partial=True)
except ValidationError as e:
    print(f"BaseModel no meio do stream: {e.errors(include_url=False)[0]['type']}")

partial = TypeAdapter(BasketStream).validate_json(cut, experimental_allow_partial=True)
print(f"BasketStream no meio do stream: {partial}")
print(f"Basket no final: {Basket.model_validate_json(full_obj)}")

BaseModel no meio do stream: missing
BasketStream no meio do stream: {'owner': 'ana', 'items': [Item(name='café', qty=2)]}
Basket no final: owner='ana' items=[Item(name='café', qty=2), Item(name='pão', qty=6)]


## 11. Performance: medindo os caminhos

Os números absolutos variam por máquina — o que importa é a **ordem de grandeza relativa**.

In [18]:
from dataclasses import dataclass

class OrderLine(BaseModel):
    sku: str
    qty: int
    price: float

@dataclass(slots=True)
class OrderLineDC:
    sku: str
    qty: int
    price: float

payload = {"sku": "ABC-123", "qty": 3, "price": 9.9}
payload_json = json.dumps(payload)
N = 50_000

batch = [payload] * 100
batch_json = json.dumps(batch)
LinesAdapter = TypeAdapter(list[OrderLine])

single = {
    "dataclass(**dict)          sem validação": lambda: OrderLineDC(**payload),
    "Model.model_construct()    sem validação": lambda: OrderLine.model_construct(**payload),
    "Model.model_validate(dict)": lambda: OrderLine.model_validate(payload),
    "Model.model_validate_json(str)": lambda: OrderLine.model_validate_json(payload_json),
    "Model.model_validate(json.loads(str))": lambda: OrderLine.model_validate(json.loads(payload_json)),
}
lots = {
    "lote 100: adapter reutilizado .validate_json": lambda: LinesAdapter.validate_json(batch_json),
    "lote 100: json.loads + [Model(**d) for d]": lambda: [OrderLine(**d) for d in json.loads(batch_json)],
    "lote 100: TypeAdapter recriado (ERRADO)": lambda: TypeAdapter(list[OrderLine]).validate_json(batch_json),
}

def bench(cases: dict, n: int) -> None:
    base = None
    for name, fn in cases.items():
        t = min(timeit.repeat(fn, number=n, repeat=3)) / n * 1e6
        base = base or t
        print(f"{name:<46} {t:9.2f} µs/op  ({t / base:5.1f}x)")

bench(single, N)
print()
bench(lots, 500)

dataclass(**dict)          sem validação            0.15 µs/op  (  1.0x)


Model.model_construct()    sem validação            1.24 µs/op  (  8.1x)


Model.model_validate(dict)                          0.59 µs/op  (  3.9x)
Model.model_validate_json(str)                      0.68 µs/op  (  4.4x)


Model.model_validate(json.loads(str))               1.64 µs/op  ( 10.8x)

lote 100: adapter reutilizado .validate_json       47.21 µs/op  (  1.0x)
lote 100: json.loads + [Model(**d) for d]          81.34 µs/op  (  1.7x)


lote 100: TypeAdapter recriado (ERRADO)            97.52 µs/op  (  2.1x)


Leitura dos resultados (o que costuma se repetir em qualquer máquina):

* `dataclass` é o piso: não faz nada além de atribuir.
* `model_validate` custa ~1 µs — **irrelevante** numa requisição HTTP (que leva ms), relevante num loop de milhões.
* ⚠️ **`model_construct` não é atalho de performance na v2.** Ele roda em Python puro, enquanto a validação roda em Rust —
  para modelos simples ele empata ou perde. Use-o só quando precisar **semanticamente** pular a validação (dado já confiável),
  nunca "para ganhar velocidade" (esse conselho vem da v1).
* Em lotes, `TypeAdapter(list[Model]).validate_json` reutilizado vence montar a lista na mão com `json.loads`:
  o parse e o loop acontecem dentro do Rust.
* Recriar o `TypeAdapter` a cada chamada paga a construção do schema toda vez — crie no nível do módulo.

## 12. Evolução de contrato (v1 → v2)

APIs públicas mudam, mas clientes antigos continuam mandando o formato velho. Estratégia comum: um modelo **canônico** (o mais novo)
e um `model_validator(mode="before")` que **migra** o payload antigo — a regra de negócio só conhece a versão atual.

In [19]:
class ContactV2(BaseModel):
    schema_version: Literal[2] = 2
    name: str
    emails: list[str]
    phone: str | None = None

    @model_validator(mode="before")
    @classmethod
    def upgrade(cls, data: Any) -> Any:
        if isinstance(data, dict) and data.get("schema_version", 1) == 1:
            data = dict(data)
            data["schema_version"] = 2
            data["emails"] = [data.pop("email")] if data.get("email") else []
            data["name"] = data.pop("nome", data.get("name"))
        return data

v1_payload = {"nome": "Ana", "email": "ana@x.com"}
v2_payload = {"schema_version": 2, "name": "Bia", "emails": ["b@x.com", "b2@x.com"]}
print(f"ContactV2.model_validate(v1_payload): {ContactV2.model_validate(v1_payload)}")
print(f"ContactV2.model_validate(v2_payload): {ContactV2.model_validate(v2_payload)}")

ContactV2.model_validate(v1_payload): schema_version=2 name='Ana' emails=['ana@x.com'] phone=None
ContactV2.model_validate(v2_payload): schema_version=2 name='Bia' emails=['b@x.com', 'b2@x.com'] phone=None


## ⚠️ Pegadinhas, efeitos colaterais e caveats

### Já demonstradas acima

| # | pegadinha | onde | como evitar |
|---|---|---|---|
| 1 | união sem discriminador escolhe por heurística | §1 | uniões **sempre** discriminadas |
| 2 | `model_construct` não é atalho de performance na v2 | §11 | usar só para pular validação de dado confiável |
| 3 | recriar `TypeAdapter` a cada chamada | §11 | adaptador no nível do módulo |

### Novas

**1. Modelo genérico sem parâmetro não valida nada.**

In [20]:
Envelope = TypeVar("Envelope")


class Wrapped(BaseModel, Generic[Envelope]):
    data: Envelope


print(f"Wrapped(data='não sou int'): {Wrapped(data='não sou int')!r}")
try:
    Wrapped[int](data="não sou int")
except ValidationError as ex:
    print(f"Wrapped[int](data=...): {ex.errors()[0]['type']}")

Wrapped(data='não sou int'): Wrapped(data='não sou int')
Wrapped[int](data=...): int_parsing


`Wrapped` sozinho trata `Envelope` como `Any`. Esquecer o `[int]` num ponto do código desliga a validação só ali.

**2. `@validate_call` valida uma cópia: mutações da função não chegam ao chamador.**

In [21]:
@validate_call
def add_default_tag(tags: list[str]) -> None:
    tags.append("padrão")


my_tags = ["urgente"]
add_default_tag(my_tags)
print(f"my_tags depois da chamada: {my_tags}")

my_tags depois da chamada: ['urgente']


Sem o decorador, `my_tags` ganharia `"padrão"`. Com ele, o argumento validado é uma lista **nova**. Adicionar `@validate_call` a uma
função existente muda o comportamento de quem dependia do efeito colateral — retorne o valor em vez de mutar o argumento.

**3. Subclasse num campo tipado pela base: a serialização usa o tipo declarado.**

In [22]:
class Animal(BaseModel):
    name: str


class Dog(Animal):
    good_boy: bool = True


class Owner(BaseModel):
    pet: Animal


owner = Owner(pet=Dog(name="Rex"))
print(f"type(owner.pet).__name__: {type(owner.pet).__name__}")
print(f"owner.model_dump(): {owner.model_dump()}")
print(f"owner.model_dump(serialize_as_any=True): {owner.model_dump(serialize_as_any=True)}")

type(owner.pet).__name__: Dog
owner.model_dump(): {'pet': {'name': 'Rex'}}
owner.model_dump(serialize_as_any=True): {'pet': {'name': 'Rex', 'good_boy': True}}


O objeto **é** um `Dog`, mas o dump mostra só os campos de `Animal` (comportamento da v2; a v1 serializava pelo tipo real). Para
polimorfismo, declare a união discriminada (`Annotated[Dog | Cat, Field(discriminator=...)]`) ou use `serialize_as_any=True`.

**4. Referência adiantada: a classe não está pronta até o tipo citado existir.**

In [23]:
class Node(BaseModel):
    child: "Leaf"


try:
    Node(child={"value": 1})
except Exception as ex:
    print(f"Node antes de Leaf: {type(ex).__name__}: {str(ex).splitlines()[0]}")


class Leaf(BaseModel):
    value: int


print(f"Node depois de Leaf: {Node(child={'value': 1})!r}")

Node antes de Leaf: PydanticUserError: `Node` is not fully defined; you should define `Leaf`, then call `Node.model_rebuild()`.
Node depois de Leaf: Node(child=Leaf(value=1))


O erro aparece no **uso**, não na definição — num módulo importado em ordem diferente, só em produção. Defina os tipos na ordem
das dependências ou chame `Node.model_rebuild()` explicitamente depois de definir `Leaf` (e teste o import isolado do módulo).

## 13. Checklist de produção

| ✔ | prática |
|---|---|
| ☐ | Entrada de API com `extra="forbid"`; DTOs separados por direção (`Create`/`Update`/`Public`) |
| ☐ | `model_validate_json` para corpos JSON; `TypeAdapter` em nível de módulo |
| ☐ | Strict em campos sensíveis (dinheiro em centavos, IDs, flags de permissão) |
| ☐ | `Decimal` para dinheiro; `datetime` sempre com timezone |
| ☐ | Uniões polimórficas **sempre** discriminadas |
| ☐ | `SecretStr` para segredos; config via `BaseSettings` com `frozen=True`, validada no startup |
| ☐ | Regras reutilizáveis como tipos `Annotated`, não copiadas entre modelos |
| ☐ | Erros com códigos estáveis (`PydanticCustomError`) quando o cliente precisa reagir |
| ☐ | Dump com alias só na fronteira; `by_alias=False` para dentro do domínio |
| ☐ | JSON Schema publicado/testado como contrato (snapshot test) |
| ☐ | Pydantic na **borda**; `dataclass`/objetos simples no núcleo quente |

### Exercícios
1. Adicione `PaymentFailed` ao webhook com `error_code: Literal[...]` e trate no `match`.
2. Crie `CursorPage[T]` (paginação por cursor) e use em `ApiResponse`.
3. Transforme `GeoPoint` num tipo com JSON Schema próprio (`__get_pydantic_json_schema__`).
4. Escreva um teste que compara `CreateTicket.model_json_schema()` com um snapshot salvo em disco.

➡️ Próximo módulo: [`../dataclass/dataclass.md`](../dataclass/dataclass.md) · Casos de uso com LLM: [`../AI_use_cases/`](../AI_use_cases/README.md)